In [1]:
import os
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_curve, auc, confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

In [2]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

PATCH_SIZE = 224
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [3]:
root_dir = "./patches_dataset/patches_v2_seed42"

pos_dir = os.path.join(root_dir, "positive")
neg_dir = os.path.join(root_dir, "negative")

def build_df_from_folders(pos_dir, neg_dir):
    rows = []

    for fname in os.listdir(pos_dir):
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            patch_id = int(os.path.splitext(fname)[0])
            rows.append({
                "patch_id": patch_id,
                "label": 1,
                "type": "positive",
            })

    for fname in os.listdir(neg_dir):
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            patch_id = int(os.path.splitext(fname)[0])
            rows.append({
                "patch_id": patch_id,
                "label": 0,
                "type": "negative",
            })

    df = pd.DataFrame(rows)
    return df

df = build_df_from_folders(pos_dir, neg_dir)

df["patch_id"] = df["patch_id"].astype(int)
df["label"] = df["label"].astype(int)
df["type"] = df["type"].astype(str)

print("dataset size:", len(df))
print(df["label"].value_counts())
print(df.head())

dataset size: 8820
label
1    5294
0    3526
Name: count, dtype: int64
   patch_id  label      type
0      3975      1  positive
1       348      1  positive
2      1804      1  positive
3      4968      1  positive
4      1810      1  positive


In [4]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,#------------------------------------------------
    random_state=SEED,
    stratify=df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

print("train/val/test:", len(train_df), len(val_df), len(test_df))
print("train label counts:\n", train_df["label"].value_counts())
print("val label counts:\n", val_df["label"].value_counts())
print("test label counts:\n", test_df["label"].value_counts())

train/val/test: 6174 1323 1323
train label counts:
 label
1    3706
0    2468
Name: count, dtype: int64
val label counts:
 label
1    794
0    529
Name: count, dtype: int64
test label counts:
 label
1    794
0    529
Name: count, dtype: int64


In [7]:
train_transform = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(
        translate_percent={"x": (-0.02, 0.02), "y": (-0.02, 0.02)},
        scale=(0.97, 1.05),
        rotate=(-10, 10),
        fit_output=False,
        p=0.6
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.10,
        contrast_limit=0.10,
        p=0.4
    ),
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(PATCH_SIZE, PATCH_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

In [8]:
class PatchDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        patch_id = int(row["patch_id"])
        label = int(row["label"])
        t = row["type"].strip().lower()

        subdir = "positive" if t == "positive" else "negative"
        img_path = os.path.join(self.root_dir, subdir, f"{patch_id}.png")

        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(image=img)["image"]

        y = torch.tensor(label, dtype=torch.float32)
        return img, y

In [9]:
batch_size = 32

train_dataset = PatchDataset(train_df, root_dir=root_dir, transform=train_transform)
val_dataset   = PatchDataset(val_df,   root_dir=root_dir, transform=eval_transform)
test_dataset  = PatchDataset(test_df,  root_dir=root_dir, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

In [10]:
backbone = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')
backbone = backbone.to(device)
backbone.eval()


Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /Users/zahra/.cache/torch/hub/main.zip


/Users/zahra/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/Users/zahra/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/Users/zahra/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /Users/zahra/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████████████████████████████████| 84.2M/84.2M [00:02<00:00, 41.1MB/s]


DinoVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 384, kernel_size=(14, 14), stride=(14, 14))
    (norm): Identity()
  )
  (blocks): ModuleList(
    (0-11): 12 x NestedTensorBlock(
      (norm1): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (attn): MemEffAttention(
        (qkv): Linear(in_features=384, out_features=1152, bias=True)
        (proj): Linear(in_features=384, out_features=384, bias=True)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (ls1): LayerScale()
      (drop_path1): Identity()
      (norm2): LayerNorm((384,), eps=1e-06, elementwise_affine=True)
      (mlp): Mlp(
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (act): GELU(approximate='none')
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (drop): Dropout(p=0.0, inplace=False)
      )
      (ls2): LayerScale()
      (drop_path2): Identity()
    )
  )
  (norm): LayerNorm((384,), eps=1e-06, elementwise_affi

In [12]:
for p in backbone.parameters():
    p.requires_grad = False

class DinoBinaryClassifier(nn.Module):
    def __init__(self, backbone, feat_dim=384):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        feats = self.backbone(x)
        logits = self.head(feats).squeeze(1)
        return logits

model = DinoBinaryClassifier(backbone, feat_dim=384).to(device)

In [14]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.head.parameters(), lr=1e-3)
THRESHOLD = 0.5

In [15]:
def train_one_epoch():
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs >= THRESHOLD).float()
        correct += (preds == y).sum().item()
        total += x.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)


@torch.no_grad()
def evaluate_scores(loader):
    model.eval()
    all_scores = []
    all_labels = []

    for x, y in loader:
        x = x.to(device)
        logits = model(x)
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_scores.append(probs)
        all_labels.append(y.numpy())

    scores = np.concatenate(all_scores, axis=0).astype(np.float64)
    labels = np.concatenate(all_labels, axis=0).astype(np.int32)
    return scores, labels


@torch.no_grad()
def evaluate_loss_acc(loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * x.size(0)

        probs = torch.sigmoid(logits)
        preds = (probs >= THRESHOLD).float()
        correct += (preds == y).sum().item()
        total += x.size(0)

    return total_loss / max(total, 1), correct / max(total, 1)

In [ ]:
epochs = 10
best_val_auc = -1.0
best_state_dict = None

for epoch in range(epochs):
    loss, acc = train_one_epoch()

    val_loss, val_acc = evaluate_loss_acc(val_loader)
    val_scores, val_labels = evaluate_scores(val_loader)

    fpr_val, tpr_val, _ = roc_curve(val_labels, val_scores)
    val_auc = auc(fpr_val, tpr_val)

    print(
        f"Epoch {epoch} | "
        f"Train Loss {loss:.4f} | Train Acc {acc:.4f} | "
        f"Val Loss {val_loss:.4f} | Val Acc {val_acc:.4f} | Val AUC {val_auc:.4f}"
    )

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)

Epoch 0 | Train Loss 0.5699 | Train Acc 0.7008 | Val Loss 0.4946 | Val Acc 0.7732 | Val AUC 0.8211
Epoch 1 | Train Loss 0.5094 | Train Acc 0.7441 | Val Loss 0.4680 | Val Acc 0.7770 | Val AUC 0.8385


In [ ]:
scores, labels = evaluate_scores(test_loader)

fpr, tpr, _ = roc_curve(labels, scores)
roc_auc = auc(fpr, tpr)

preds = (scores >= THRESHOLD).astype(np.int32)

print("\nTEST RESULTS")
print("ROC AUC:", roc_auc)
print("Confusion matrix:\n", confusion_matrix(labels, preds))
print(classification_report(labels, preds, digits=4))

plt.figure()
plt.plot(fpr, tpr, label=f"ROC AUC={roc_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC")
plt.legend()
plt.show()